In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import gym
from gym import spaces
# import torch
# import torch.nn as nn
# import torch.optim as optim
# import torch.nn.functional as F
# from torch.distributions import Categorical, MultivariateNormal
import pickle
import os
import random
import math
from tqdm import trange

# import plotly.express as px
# import plotly.io as pio
# import plotly.graph_objects as go
# import dash
# from dash import dcc, html
# from dash.dependencies import Input, Output
# pio.renderers.default = "notebook"

import heapq
import json
import time
# 假設的 DEVICE
# DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# print(DEVICE)

## env

In [ ]:
class SimpleWBEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    # ==========================================================
    # init
    # ==========================================================
    def __init__(
        self,
        wip_path: str,
        mach_path: str,
        max_lots: int = 100,
        max_groups: int = 20,
        reward_mode: str = "tardiness",
        compat_pkl_path: str = None,
        changeover_time: float = 0.25,
        changeover_penalty: float = 0.0,
        state_delay_hours: float = 0.0,
        exclusive_group_threshold: int = 10,
    ):
        super().__init__()

        # ---------------- basic params ----------------
        self.BASE_START_DATE = datetime(2026, 1, 1)
        self.changeover_time = float(changeover_time)
        self.changeover_penalty_value = float(changeover_penalty)
        self.state_delay_hours = float(state_delay_hours)
        self.exclusive_group_threshold = int(exclusive_group_threshold)
        
        # ---------------- load data ----------------
        wip = pd.read_csv(wip_path).iloc[:max_lots].reset_index(drop=True)
        mach = pd.read_csv(mach_path)

        self.num_lots = len(wip)
        if self.num_lots == 0:
            raise ValueError("wip.csv 內沒有資料")

        # ---------------- machine groups ----------------
        self.group_ids = mach["EQP_GROUP_ID"].unique().tolist()[:max_groups]
        self.num_groups = len(self.group_ids)

        # ---------------- cache WIP ----------------
        self.process_time = wip["PROD_TIME"].fillna(0.0).to_numpy(float)

        self.lot_recipe_cache = (
            wip["RECIPENAME"].fillna("").astype(str).to_numpy()
            if "RECIPENAME" in wip.columns
            else np.array([""] * self.num_lots, dtype=object)
        )

        # SOD → due hours
        wip["SOD"] = pd.to_datetime(wip["SOD"], errors="coerce")
        cutoff = self.BASE_START_DATE + timedelta(days=3)
        wip.loc[wip["SOD"] <= cutoff, "SOD"] += pd.Timedelta(days=0)

        due_hours = []
        for sod in wip["SOD"]:
            if pd.isna(sod):
                due_hours.append(0.0)
            else:
                d = sod.to_pydatetime() - timedelta(days=2)
                due_hours.append((d - self.BASE_START_DATE).total_seconds() / 3600.0)
        self.due_hours = np.asarray(due_hours, dtype=float)
        
        # ---------------- lot release ----------------
        if "STATE" in wip.columns:
            self.lot_release_hours = np.array(
                [
                    self.state_delay_hours if str(s).strip().upper() == "X" else 0.0
                    for s in wip["STATE"]
                ],
                dtype=float,
            )
            self.state_X_flag = np.array(
                [
                    1.0 if str(s).strip().upper() == "X" else 0.0
                    for s in wip["STATE"]
                ],
                dtype=np.float32,
            )
        else:
            self.lot_release_hours = np.zeros(self.num_lots, dtype=float)
            self.state_X_flag = np.zeros(self.num_lots, dtype=np.float32)
            
        self.urgent_flag = (self.due_hours <= self.process_time).astype(np.float32)
        
        # ---------------- group info ----------------
        self.group_mcount = np.ones(self.num_groups, dtype=float)
        self.group_arrive = np.zeros(self.num_groups, dtype=float)
        self.group_init_recipe = np.array([""] * self.num_groups, dtype=object)

        for i, gid in enumerate(self.group_ids):
            sub = mach[mach["EQP_GROUP_ID"] == gid]
            if "m_count" in sub.columns:
                self.group_mcount[i] = max(float(sub["m_count"].iloc[0]), 1.0)
            if "Arrive" in sub.columns:
                self.group_arrive[i] = max(float(sub["Arrive"].iloc[0]), 0.0)
            if "NEW_RECIPE" in sub.columns:
                r = sub["NEW_RECIPE"].iloc[0]
                self.group_init_recipe[i] = "" if pd.isna(r) else str(r)

        # ---------------- normalization ----------------
        self.norm_due = max(self.due_hours.max(), 1.0)
        self.norm_proc = max(self.process_time.max(), 1.0)
        self.norm_mcount = max(self.group_mcount.max(), 1.0)

        # ---------------- compat cache ----------------
        self.compat_dict = None
        if compat_pkl_path is not None:
            try:
                with open(compat_pkl_path, "rb") as f:
                    self.compat_dict = pickle.load(f)
            except Exception:
                self.compat_dict = None

        self.allowed_group_cache = [
            self._compute_allowed_groups(i, wip) for i in range(self.num_lots)
        ]

        # ---------------- urgency order (exclusive → slack → LOCATION) ----------------
        
        # slack：越小越急
        slack = self.due_hours - self.process_time
        
        # 每個 lot 能做的機群數量
        allowed_counts = np.array(
            [len(a) for a in self.allowed_group_cache], dtype=int
        )
        
        # 是否為「專機 / 稀缺 lot」
        is_exclusive = allowed_counts <= self.exclusive_group_threshold
        
        # exclusive priority：0 先排，1 後排
        exclusive_priority = (~is_exclusive).astype(int)
        
        # ---------------- LOCATION priority ----------------
        if "LOCATION" in wip.columns:
            def loc_priority(x):
                if pd.isna(x) or str(x).strip() == "":
                    return 4   # 最後
                x = str(x).strip().upper()
                if x == "K3-3F":
                    return 0
                elif x == "K3-1F":
                    return 1
                elif x == "K3-2F":
                    return 2
                else:
                    return 3   # 其他非空
        
            location_priority = np.array(
                [loc_priority(x) for x in wip["LOCATION"]],
                dtype=int,
            )
        else:
            location_priority = np.zeros(self.num_lots, dtype=int)
        
        # ---------------- lexsort ----------------
        # 最後一個 key 最重要
        self.urgency_order = np.lexsort(
            (
                location_priority,   # 第三順位（LOCATION）
                slack,               # 第二順位（時間急迫性）
                exclusive_priority,  # 第一順位（專機）
            )
        )

        # ---------------- spaces (dummy) ----------------
        self.observation_space = spaces.Box(
            low=0.0, high=1.0, shape=(1,), dtype=np.float32
        )
        self.action_space = spaces.Discrete(self.num_lots * self.num_groups)

        # ---------------- episode state ----------------
        self.reward_mode=reward_mode
        self.w1 = 0.1
        self.w2 = 0.9
        self.reset()

    # ==========================================================
    # reset
    # ==========================================================
    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.schedule_log = []
        self.schedule_step = 0
        self.urgency_ptr = 0

        self.group_available = self.group_arrive.copy()
        self.group_recipe = self.group_init_recipe.copy()
        self.done_lot = np.zeros(self.num_lots, dtype=np.int8)
        self.finish_time = np.zeros(self.num_lots, dtype=float)
        self.group_scheduled_count = np.zeros(self.num_groups, dtype=np.float32)

        return None, {}

    # ==========================================================
    # step
    # ==========================================================
    def step(self, action):
        lot_idx = action // self.num_groups
        group_idx = action % self.num_groups

        if lot_idx >= self.num_lots or group_idx >= self.num_groups:
            return None, -200.0, False, False, {}

        if self.done_lot[lot_idx]:
            return None, -100.0, False, False, {}

        base_proc = self.process_time[lot_idx]
        m = self.group_mcount[group_idx]
        effective_proc = base_proc / m

        lot_recipe = self.lot_recipe_cache[lot_idx]
        need_change = (lot_recipe != self.group_recipe[group_idx])

        change_t = self.changeover_time * m if need_change else 0.0
        change_penalty = self.changeover_penalty_value * m if need_change else 0.0

        avail = max(self.group_available[group_idx], self.lot_release_hours[lot_idx])
        start = avail + change_t
        finish = start + effective_proc

        self.group_available[group_idx] = finish
        self.group_recipe[group_idx] = lot_recipe
        self.done_lot[lot_idx] = 1
        self.finish_time[lot_idx] = finish
        self.schedule_step += 1
        self.group_scheduled_count[group_idx] += 1.0
        
        due = self.due_hours[lot_idx]
        tardiness = max(0.0, finish - due)
        r_tardiness = -tardiness
        setup_time = m if need_change else 0
        r_setup = -setup_time*0.25
        
        if self.reward_mode == "tardiness":
            reward = r_tardiness 
        elif self.reward_mode == "setup":
            reward = r_setup
        elif self.reward_mode == "linear":
            reward = self.w1*r_tardiness +self.w2*r_setup
            

        self.schedule_log.append({
            "lot_id": int(lot_idx),
            "group_id": int(group_idx),
            "start": float(start),
            "finish": float(finish),
            "due_times": (self.BASE_START_DATE + timedelta(hours=due)).strftime("%Y/%m/%d"),
            "process_time": float(base_proc),
            "effective_proc_time": float(effective_proc),
            "reward": float(reward),
            "tardiness": float(tardiness),
        })

        

        done = self.done_lot.sum() >= self.num_lots
        return None, reward, done, False, {"tardiness": float(tardiness),"setup": setup_time*2}

    # ==========================================================
    # urgency
    # ==========================================================
    def get_most_urgent_lot(self):
        while self.urgency_ptr < self.num_lots:
            i = self.urgency_order[self.urgency_ptr]
            if not self.done_lot[i]:
                return int(i)
            self.urgency_ptr += 1
        return None

    # ==========================================================
    # allowed groups (cached)
    # ==========================================================
    def _compute_allowed_groups(self, lot_idx, wip):
        if self.compat_dict is None:
            return list(range(self.num_groups))

        lot_no = (
            wip.loc[lot_idx, "TT_LOT_NO"]
            if "TT_LOT_NO" in wip.columns
            else lot_idx
        )

        keys = [(lot_no, 2800), lot_no, (str(lot_no), 2800)]
        cand = None
        for k in keys:
            if k in self.compat_dict:
                cand = self.compat_dict[k]
                break

        if cand is None:
            return list(range(self.num_groups))

        cand = {str(x) for x in cand}
        allowed = []
        for i, gid in enumerate(self.group_ids):
            s = str(gid)
            if s in cand or any(s in c or c in s for c in cand):
                allowed.append(i)

        return allowed if allowed else list(range(self.num_groups))

    # ==========================================================
    # features (FAST)
    # ==========================================================
    def get_features_for_lot(self, lot_idx):
        allowed = self.allowed_group_cache[lot_idx]
        allowed_count = len(allowed)
        allowed_ratio = allowed_count / self.num_groups
        base_proc = self.process_time[lot_idx]
        due = self.due_hours[lot_idx]
        lot_recipe = self.lot_recipe_cache[lot_idx]
        lot_release = self.lot_release_hours[lot_idx]
        urgent_flag = self.urgent_flag[lot_idx]

        state_X = self.state_X_flag[lot_idx]

        step_norm = self.schedule_step / max(1.0, self.num_lots)

        feats = []
        for g in allowed:
            m = self.group_mcount[g]
            need_change = (lot_recipe != self.group_recipe[g])
            change_t = self.changeover_time * m if need_change else 0.0

            pred_start = max(self.group_available[g], lot_release)
            pred_finish = pred_start + change_t + base_proc / m

            group_load_ratio = self.group_scheduled_count[g] / self.num_lots
            
            phi = np.array([
                1.0,
                due / self.norm_due,
                base_proc / self.norm_proc,
                urgent_flag,
                state_X, 
                allowed_ratio,
                m / self.norm_mcount,
                self.group_available[g] / self.norm_due,
                (due - pred_finish) / self.norm_due,
                1.0 if not need_change else 0.0,
                step_norm,
            ], dtype=np.float32)

            feats.append((g, phi))

        return feats

    # ==========================================================
    def render(self, mode="human"):
        pass


## LinUCB &LinTS

In [ ]:
import numpy as np

class LinearBanditAgent:
    """
    Per-arm linear bandit agent (Disjoint Linear Bandit).

    Each arm maintains its own:
      - A     : (d, d)
      - b     : (d, 1)
      - A_inv : (d, d)

    Parameters
    ----------
    num_arms : int
        Number of arms (e.g., machine groups)
    n_features : int
        Feature dimension
    alpha : float
        Exploration parameter (LinUCB / LinTS)
    algorithm : str
        'LinUCB', 'LinTS', or 'Greedy'
    reg : float
        Ridge regularization lambda
    """

    def __init__(
        self,
        num_arms: int,
        n_features: int,
        alpha: float = 1.0,
        algorithm: str = "LinUCB",
        reg: float = 1.0,
    ):
        self.num_arms = int(num_arms)
        self.n_features = int(n_features)
        self.alpha = float(alpha)
        self.algorithm = algorithm
        self.reg = float(reg)

        # Per-arm matrices
        self.A = np.array(
            [self.reg * np.eye(self.n_features) for _ in range(self.num_arms)]
        )  # (K, d, d)

        self.b = np.array(
            [np.zeros((self.n_features, 1)) for _ in range(self.num_arms)]
        )  # (K, d, 1)

        self.A_inv = np.array(
            [np.linalg.inv(self.A[a]) for a in range(self.num_arms)]
        )  # (K, d, d)

    # -------------------------------------------------
    # utilities
    # -------------------------------------------------
    def _ensure_x_shape(self, x):
        """
        Ensure x is column vector with shape (d, 1)
        """
        x = np.asarray(x, dtype=float)
        if x.ndim == 1:
            x = x.reshape(-1, 1)
        elif x.ndim == 2 and x.shape[1] == 1:
            pass
        elif x.ndim == 2 and x.shape[0] == 1:
            x = x.reshape(-1, 1)
        else:
            x = x.reshape(self.n_features, 1)
        return x

    # -------------------------------------------------
    # scoring
    # -------------------------------------------------
    def get_score(self, arm_idx: int, x):
        """
        Compute score for a specific arm.

        Returns
        -------
        float
            UCB / TS / Greedy score
        """
        if arm_idx < 0 or arm_idx >= self.num_arms:
            raise IndexError("arm_idx out of range")

        x = self._ensure_x_shape(x)           # (d, 1)
        Ainv = self.A_inv[arm_idx]            # (d, d)
        b = self.b[arm_idx]                   # (d, 1)

        theta_hat = Ainv @ b                  # (d, 1)

        # IMPORTANT: use .item() to avoid NumPy 1.25 warning
        mean_pred = (theta_hat.T @ x).item()
        var = (x.T @ Ainv @ x).item()

        if self.algorithm == "LinUCB":
            bonus = self.alpha * np.sqrt(var)
            return mean_pred + bonus

        elif self.algorithm == "LinTS":
            std = self.alpha * np.sqrt(var)
            return np.random.normal(loc=mean_pred, scale=std)

        else:  # Greedy
            return mean_pred

    # -------------------------------------------------
    # update
    # -------------------------------------------------
    def update(self, arm_idx: int, x, reward: float):
        """
        Update only the selected arm using Sherman–Morrison.
        """
        if arm_idx < 0 or arm_idx >= self.num_arms:
            raise IndexError("arm_idx out of range")

        x = self._ensure_x_shape(x)            # (d, 1)
        Ainv = self.A_inv[arm_idx]             # (d, d)

        # denom = 1 + x^T A_inv x
        denom = 1.0 + (x.T @ Ainv @ x).item()

        if denom <= 0.0 or not np.isfinite(denom):
            # numerical fallback
            self.A[arm_idx] += x @ x.T
            self.b[arm_idx] += reward * x
            self.A_inv[arm_idx] = np.linalg.inv(self.A[arm_idx])
            return

        # Sherman–Morrison update
        num = (Ainv @ x) @ (x.T @ Ainv)        # (d, d)
        self.A_inv[arm_idx] = Ainv - num / denom

        # Keep A, b in sync
        self.A[arm_idx] += x @ x.T
        self.b[arm_idx] += reward * x

    # -------------------------------------------------
    # diagnostics
    # -------------------------------------------------
    def get_theta(self, arm_idx: int):
        """
        Return theta_hat for a given arm as 1D array.
        """
        if arm_idx < 0 or arm_idx >= self.num_arms:
            raise IndexError("arm_idx out of range")
        return (self.A_inv[arm_idx] @ self.b[arm_idx]).reshape(-1)

    def reset(self):
        """
        Reset all arms (useful between experiments).
        """
        self.A = np.array(
            [self.reg * np.eye(self.n_features) for _ in range(self.num_arms)]
        )
        self.b = np.array(
            [np.zeros((self.n_features, 1)) for _ in range(self.num_arms)]
        )
        self.A_inv = np.array(
            [np.linalg.inv(self.A[a]) for a in range(self.num_arms)]
        )


## Hyperparameter

In [ ]:
# --- 1. 環境參數 ---
WIP_FILE_PATH = r"wip.csv"
MACH_FILE_PATH = r"mach.csv"
CHECKPOINT_PATH = r"checkpoints\cp1"

num_lot = 1000
num_group = 150
#1067 203
# 訓練控制參數
N_FEATURES = 11  # 對應 get_features 中的特徵數量
ALPHA = 0.9  # 探索參數 
N_EPISODES = 500 # 訓練回合數 
penalty = 0

# LinUCB train

In [ ]:
# env 與 agent 
env = SimpleWBEnv(wip_path=WIP_FILE_PATH, 
                  mach_path=MACH_FILE_PATH, 
                  #compat_pkl_path = COMPAT_PKL_PATH,
                  reward_mode="linear",
                  max_lots = num_lot, 
                  max_groups=num_group, 
                 changeover_penalty = penalty)
env.w1 = 0.8
env.w2 = 0.2

agent_ucb = LinearBanditAgent(n_features=N_FEATURES, num_arms = env.num_groups , alpha=ALPHA, algorithm='LinUCB')

In [ ]:
def save_agent(agent:LinearBanditAgent, path:str):
    """
    Save agent internals to .npz
    """
    npz_dict = {
        "num_arms": agent.num_arms,
        "n_features": agent.n_features,
        "alpha": agent.alpha,
        "algorithm": agent.algorithm,
        "reg": agent.reg,
        "A": agent.A,
        "b": agent.b,
        "A_inv": agent.A_inv
    }
    np.savez_compressed(path, **npz_dict)

def load_agent(path:str) -> LinearBanditAgent:
    d = np.load(path, allow_pickle=True)
    agent = LinearBanditAgent(int(d["num_arms"]), int(d["n_features"]), alpha=float(d["alpha"]), algorithm=str(d["algorithm"]), reg=float(d["reg"]))
    agent.A = d["A"]
    agent.b = d["b"]
    agent.A_inv = d["A_inv"]
    return agent

In [ ]:
def train(env,
          agent: LinearBanditAgent,
          num_episodes: int = 40,
          verbose: bool = True,
          save_dir: str = "checkpoints",
          top_k: int = 1,
          start_saving_after: int = None,
          checkpoint_name: str = "saved_checkpoints.json"):

    os.makedirs(save_dir, exist_ok=True)
    if start_saving_after is None:
        start_saving_after = num_episodes - top_k - 10

    best_heap = []
    history = {
        "episode_rewards": [],
        "episode_tardiness": [],
        "episode_setup": [],
        "steps_per_episode": [],
    }

    for ep in range(num_episodes):
        _, _ = env.reset()
        done = False
        ep_reward = 0.0
        ep_tard = 0.0
        ep_setup = 0.0
        steps = 0

        # safety cap（理論上不會碰到）
        cap = env.num_lots * env.num_groups * 2

        while not done and steps < cap:
            lot_idx = env.get_most_urgent_lot()
            if lot_idx is None:
                break

            # === FAST: list[(group_id, phi)] ===
            feats = env.get_features_for_lot(lot_idx)

            if not feats:
                env.done_lot[lot_idx] = 1
                env.schedule_log.append({
                    "lot_id": int(lot_idx),
                    "group_id": -1,
                    "note": "no_compatible_group"
                })
                continue

            # === score ===
            best_score = -np.inf
            best_group = None
            best_phi = None

            for group_id, phi in feats:
                s = agent.get_score(group_id, phi)
                if s > best_score:
                    best_score = s
                    best_group = group_id
                    best_phi = phi

            if best_group is None:
                env.done_lot[lot_idx] = 1
                continue

            action = int(lot_idx * env.num_groups + best_group)

            _, reward, done, _, info = env.step(action)
            agent.update(best_group, best_phi.reshape(-1, 1), float(reward))

            ep_reward += float(reward)
            ep_tard += float(info.get("tardiness", 0.0))
            ep_setup += float(info.get("setup", 0.0))
            steps += 1


        history["episode_rewards"].append(ep_reward)
        history["episode_tardiness"].append(ep_tard)
        history["steps_per_episode"].append(steps)

        # ================= checkpoint =================
        if (ep + 1) >= start_saving_after:
            if len(best_heap) < top_k or ep_reward > best_heap[0][0]:
                ts = int(time.time())
                base = f"ep{ep+1:03d}_r{ep_reward:.3f}_{ts}"
                agent_path = os.path.join(save_dir, base + "_agent.npz")
                sched_path = os.path.join(save_dir, base + "_schedule.csv")

                save_agent(agent, agent_path)
                pd.DataFrame(env.schedule_log).to_csv(sched_path, index=False)

                heapq.heappush(best_heap, (ep_reward, ts, agent_path, sched_path, ep+1))
                if len(best_heap) > top_k:
                    _, _, ap, sp, _ = heapq.heappop(best_heap)
                    if os.path.exists(ap):
                        os.remove(ap)
                    if os.path.exists(sp):
                        os.remove(sp)

        if verbose and ((ep + 1) % 10 == 0 or ep == 0):
            print(
                f"Episode {ep+1}/{num_episodes} | "
                f"Reward {ep_reward:.3f} | "
                f"Tardiness {ep_tard:.3f} | "
                f"Setup {ep_setup:.1f} | "
                f"Steps {steps}"
            )

    # ================= finalize index =================
    best_list = sorted(best_heap, key=lambda x: x[0], reverse=True)
    index_path = os.path.join(save_dir, checkpoint_name)

    idx_info = []
    for reward, ts, agent_path, sched_path, epnum in best_list:
        idx_info.append({
            "episode": int(epnum),
            "reward": float(reward),
            "time": int(ts),
            "agent_path": agent_path,
            "schedule_path": sched_path
        })

    with open(index_path, "w") as f:
        json.dump(idx_info, f, indent=2)

    return history, idx_info


In [ ]:
history_ucb_record = []

for i in range(4):
    env = SimpleWBEnv(wip_path=WIP_FILE_PATH, 
                  mach_path=MACH_FILE_PATH, 
                  #compat_pkl_path = COMPAT_PKL_PATH,
                  reward_mode="linear",
                  max_lots = num_lot, 
                  max_groups=num_group, 
                 changeover_penalty = penalty)
    env.w1 = 0.8
    env.w2 = 0.2

    agent_ucb = LinearBanditAgent(n_features=N_FEATURES, num_arms = env.num_groups , alpha=ALPHA*(0.95**i), algorithm='LinUCB')
    history_ucb, idx_info_ucb = train(env, agent_ucb, num_episodes=200,
                                         save_dir=f"{CHECKPOINT_PATH}_ucb_9tardiness_1setup_{i}",top_k=3,
                                         checkpoint_name=f"linUCB_l{num_lot}_m{num_group}_ep{N_EPISODES}.json")
    history_ucb_record.append(history_ucb)

In [ ]:
history_ucb, idx_info_ucb = train(env, agent_ucb, num_episodes=N_EPISODES,
                                         save_dir=f"{CHECKPOINT_PATH}_ucb_9tardiness_1setup_1",top_k=5,
                                         checkpoint_name=f"linUCB_l{num_lot}_m{num_group}_ep{N_EPISODES}.json")

In [ ]:
history_ucb["episode_rewards"]

In [ ]:
# 載第一個 best
best = idx_info_ucb[0]
agent_loaded = load_agent(best["agent_path"])
sched_df_ucb = pd.read_csv(best["schedule_path"])
print("Best episode:", best["episode"], "reward:", best["reward"])
print(sched_df_ucb)

## Result

In [ ]:
import matplotlib.pyplot as plt

for i in range(4):
    plt.figure(figsize=(8,5))
    plt.plot(history_ucb_record[i]["episode_rewards"], label="Episode reward")
    plt.xlabel("Episode")
    plt.ylabel("Reward")
    plt.title("LinUCB Training Reward over Episodes")
    plt.grid(True)
    plt.legend()

    plt.figure(figsize=(8,5))
    plt.plot(history_ucb_record[i]["episode_tardiness"], label="Episode_tardiness", color='red')
    plt.xlabel("Episode")
    plt.ylabel("Tardiness")
    plt.title("LinUCB Training Tardiness over Episodes")
    plt.grid(True)
    plt.legend()
    plt.show()


In [ ]:
def plot_gantt_by_recipe(schedule_log,
                         env=None,
                         base_date=None,
                         x_days=8,
                         last_n=None,
                         hover_cols=None,
                         title=None):
    """
    Plot interactive Gantt with Plotly, color by RECIPENAME.

    Args:
        schedule_log: list of dicts (每個 dict 至少要有 'lot_id','group_id','start','finish')
        env: SimpleWBEnv instance (optional) — 用來取得 RECIPENAME 與預設 BASE_START_DATE
        base_date: "YYYY-MM-DD" string or datetime
        x_days: x-axis 範圍天數
        last_n: 若給 int，僅畫最後 n 筆 log
        hover_cols: list of欄位名顯示在 hover tooltip (預設 ['lot_id','process_time','effective_proc_time','reward','tardiness'])
    """

    # ---------- 防呆 ----------
    if isinstance(schedule_log, SimpleWBEnv):
        schedule_log = schedule_log.schedule_log

    if not isinstance(schedule_log, (list, tuple)):
        raise TypeError(f"schedule_log 必須是 list[dict]，但收到 {type(schedule_log)}")

    if len(schedule_log) == 0:
        raise ValueError("schedule_log 為空")

    df = pd.DataFrame(schedule_log)

    # ---------- 只取最後 n 筆 ----------
    if last_n is not None:
        df = df.tail(last_n).reset_index(drop=True)

    # ---------- base date ----------
    if base_date is None and env is not None:
        base_dt = env.BASE_START_DATE
    elif isinstance(base_date, str):
        base_dt = datetime.strptime(base_date, "%Y-%m-%d")
    elif isinstance(base_date, datetime):
        base_dt = base_date
    else:
        base_dt = datetime.now()

    # ---------- recipe ----------
    if env is not None and hasattr(env, "lot_recipe_cache"):
        recipe_map = {i: r for i, r in enumerate(env.lot_recipe_cache)}
        df["RECIPENAME"] = df["lot_id"].map(recipe_map)
    else:
        if "RECIPENAME" not in df.columns:
            df["RECIPENAME"] = "Unknown"

    # ---------- datetime ----------
    df["start_dt"] = df["start"].apply(lambda h: base_dt + timedelta(hours=float(h)))
    df["finish_dt"] = df["finish"].apply(lambda h: base_dt + timedelta(hours=float(h)))
    df["group_str"] = df["group_id"].astype(str)


    # ---------- y 軸順序 ----------
    if env is not None:
        ordered_groups = [str(i) for i in range(env.num_groups - 1, -1, -1)]
    else:
        ordered_groups = sorted(df["group_str"].unique(), key=int)

    # ---------- hover 欄位 ----------
    if hover_cols is None:
        hover_cols = ["lot_id","process_time","effective_proc_time","reward","tardiness","due_times"]

    hover_dict = {col: True for col in hover_cols}
    # group_str、start_dt、finish_dt 不用顯示
    hover_dict.update({"group_str": False, "start_dt": False, "finish_dt": False})


    title_name = "Scheduling Gantt Chart (by RECIPENAME)" if title is None else title
    # ---------- Plotly timeline ----------
    fig = px.timeline(
        df,
        x_start="start_dt",
        x_end="finish_dt",
        y="group_str",
        color="RECIPENAME",
        hover_data=hover_dict,
        title=title_name
    )

    fig.update_yaxes(
        categoryorder="array",
        categoryarray=ordered_groups,
        autorange="reversed",
        title="Machine Group"
    )

    fig.update_xaxes(
        title="Time",
        range=[base_dt, base_dt + timedelta(days=x_days)]
    )

    fig.update_layout(legend_title_text="Recipe")

    fig.show()


In [ ]:
#gantt plot for LinUCB
plot_gantt_by_recipe(env.schedule_log, env=env, base_date="2026-01-01", x_days=8, title="2026-01-01 00:00 scheduling gantt with LinUCB")

## LinTS

In [ ]:
# env 與 agent 
env2 = SimpleWBEnv(wip_path=WIP_FILE_PATH, 
                  mach_path=MACH_FILE_PATH, 
                  #compat_pkl_path = COMPAT_PKL_PATH,
                  reward_mode="linear",
                  max_lots = num_lot, 
                  max_groups=num_group, 
                 changeover_penalty = penalty)
agent_ts = LinearBanditAgent(n_features=N_FEATURES,  num_arms = env2.num_groups,alpha=0.6*ALPHA, algorithm='LinTS')

In [ ]:
env2.w1 = 0.8
env2.w2 = 0.2

In [ ]:
history_ts_record = []

for i in range(2):
    env_2 = SimpleWBEnv(wip_path=WIP_FILE_PATH, 
                  mach_path=MACH_FILE_PATH, 
                  #compat_pkl_path = COMPAT_PKL_PATH,
                  reward_mode="linear",
                  max_lots = num_lot, 
                  max_groups=num_group, 
                 changeover_penalty = penalty)
    env_2.w1 = 0.8
    env_2.w2 = 0.2

    agent_ts = LinearBanditAgent(n_features=N_FEATURES,  num_arms = env_2.num_groups,alpha=0.6*ALPHA, algorithm='LinTS')
    history_ts, idx_info_ts = train(env_2, agent_ts, num_episodes=400,
                                         save_dir=f"{CHECKPOINT_PATH}_ts_8tardiness_2setup_{i}",top_k=10,
                                         checkpoint_name=f"linTS_l{num_lot}_m{num_group}_ep{N_EPISODES}.json")
    history_ts_record.append(history_ts)

In [ ]:
# 載第一個 best
best = idx_info_ts[0]
agent_loaded = load_agent(best["agent_path"])
sched_df_ts = pd.read_csv(best["schedule_path"])
print("Best episode:", best["episode"], "reward:", best["reward"])
print(sched_df_ts)

In [ ]:
history_ts["episode_rewards"]

In [ ]:
import matplotlib.pyplot as plt

for i in range(2):
    plt.figure(figsize=(8,5))
    plt.plot(history_ts_record[i]["episode_rewards"], label="Episode reward")
    plt.xlabel("Episode")
    plt.ylabel("Reward")
    plt.title("LinTS Training Reward over Episodes")
    plt.grid(True)
    plt.legend()

    plt.figure(figsize=(8,5))
    plt.plot(history_ts_record[i]["episode_tardiness"], label="Episode_tardiness", color='red')
    plt.xlabel("Episode")
    plt.ylabel("Tardiness")
    plt.title("LinTS Training Tardiness over Episodes")
    plt.grid(True)
    plt.legend()
    plt.show()


In [ ]:
#gantt plot for LinTS
plot_gantt_by_recipe(env2.schedule_log, env=env2, base_date="2026-01-01", x_days=8, title="2026-01-01 00:00 scheduling gantt with LinTS")

## Load agent and schedule

In [ ]:
# idx_info 可能從 JSON 讀取
import json
with open(f"{CHECKPOINT_PATH}/linUCB_l{num_lot}_m{num_group}_ep{N_EPISODES}.json") as f:
    idx_info = json.load(f)

# 讀取第一個最佳 schedule
best_sched_path = idx_info[0]["schedule_path"]
schedule_df = pd.read_csv(best_sched_path)

print(schedule_df.head())

In [ ]:
#LinUCB load
# ---------- 1. 讀取 saved_checkpoints.json ----------
save_dir = "checkpoints"
index_path = f"{CHECKPOINT_PATH}/linUCB_l{num_lot}_m{num_group}_ep{N_EPISODES}.json"  # 或你的 checkpoint_name

with open(index_path, "r") as f:
    idx_info_ucb = json.load(f)

# 取 reward 最好的 checkpoint（idx_info 是由大到小排的話）
best = idx_info_ucb[0]

print("Best episode:", best["episode"])
print("Reward:", best["reward"])
print("Agent path:", best["agent_path"])
print("Schedule path:", best["schedule_path"])

# ---------- 2. 載入最佳 agent（如果你想重用 agent 做其他評估） ----------
agent_loaded = load_agent(best["agent_path"])  # 你之前定義的 load_agent()

# ---------- 3. 讀取最佳排程結果 ----------
sched_df_ucb = pd.read_csv(best["schedule_path"])

# 如果 schedule_log 是 list of dict，你可以：
schedule_log = sched_df_ucb.to_dict(orient="records")

# ---------- 4. 畫甘特圖 ----------
# plot_gantt_by_recipe(schedule_log, env=env)  # 也可以傳 base_date, x_days, last_n



In [ ]:
# idx_info 可能從 JSON 讀取
import json
with open(f"{CHECKPOINT_PATH}/linTS_l{num_lot}_m{num_group}_ep{N_EPISODES}.json") as f:
    idx_info = json.load(f)

# 讀取第一個最佳 schedule
best_sched_path = idx_info[0]["schedule_path"]
schedule_df = pd.read_csv(best_sched_path)

print(schedule_df.head())

In [ ]:
#LinUCB load
# ---------- 1. 讀取 saved_checkpoints.json ----------
save_dir = "checkpoints"
index_path = f"{CHECKPOINT_PATH}/linTS_l{num_lot}_m{num_group}_ep{N_EPISODES}.json"  # 或你的 checkpoint_name

with open(index_path, "r") as f:
    idx_info_ucb = json.load(f)

# 取 reward 最好的 checkpoint（idx_info 是由大到小排的話）
best = idx_info_ucb[0]

print("Best episode:", best["episode"])
print("Reward:", best["reward"])
print("Agent path:", best["agent_path"])
print("Schedule path:", best["schedule_path"])

# ---------- 2. 載入最佳 agent（如果你想重用 agent 做其他評估） ----------
agent_loaded = load_agent(best["agent_path"])  # 你之前定義的 load_agent()

# ---------- 3. 讀取最佳排程結果 ----------
sched_df_ts = pd.read_csv(best["schedule_path"])

# 如果 schedule_log 是 list of dict，你可以：
schedule_log = sched_df_ts.to_dict(orient="records")

# ---------- 4. 畫甘特圖 ----------
plot_gantt_by_recipe(schedule_log, env=env)  # 也可以傳 base_date, x_days, last_n

